In [1]:
#모델 클래스 코드
import timm
import torch
import torch.nn as nn

class FFTBranch(nn.Module):
    def __init__(self, out_dim=256):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.fc = nn.Linear(256, out_dim)

    def forward(self, x):
        return self.fc(self.conv(x).flatten(1))


class DualStreamDetector(nn.Module):
    def __init__(self, freeze_ratio=0.7, fft_dim=256):
        super().__init__()
        self.rgb_backbone = timm.create_model(
            'efficientnet_b4', pretrained=False, num_classes=0)
        self.fft_branch   = FFTBranch(out_dim=fft_dim)
        fusion_dim = self.rgb_backbone.num_features + fft_dim  # 2048
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 512), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(512, 128),        nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, rgb, fft):
        rgb_feat = self.rgb_backbone(rgb)
        fft_feat = self.fft_branch(fft)
        fused    = torch.cat([rgb_feat, fft_feat], dim=1)
        return self.classifier(fused).squeeze(1)

In [ ]:
#모델 로드 방법
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = DualStreamDetector().to(device)
ckpt   = torch.load('best_dualstream_final.pt', map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()

In [ ]:
#추론 방법
import numpy as np
from torchvision import transforms
from PIL import Image

# RGB 전처리
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

def compute_fft(img_pil):
    """PIL Image → FFT tensor (1, 224, 224)"""
    gray     = img_pil.convert('L').resize((224, 224))
    gray_np  = np.array(gray).astype(np.float32) / 255.0
    fft      = np.fft.fftshift(np.fft.fft2(gray_np))
    magnitude = np.log(np.abs(fft) + 1e-8)
    mag_min, mag_max = magnitude.min(), magnitude.max()
    magnitude = (magnitude - mag_min) / (mag_max - mag_min + 1e-8)
    return torch.from_numpy(magnitude).unsqueeze(0).float()

def predict(image_path):
    img       = Image.open(image_path).convert('RGB')
    rgb       = val_tf(img).unsqueeze(0).to(device)       # (1,3,224,224)
    fft       = compute_fft(img).unsqueeze(0).to(device)  # (1,1,224,224)
    with torch.no_grad():
        prob_real = torch.sigmoid(model(rgb, fft)).item()
    prob_fake = 1 - prob_real
    return {
        'prob_real': round(prob_real, 4),  # sigmoid > 0.5 → REAL
        'prob_fake': round(prob_fake, 4),  # sigmoid < 0.5 → FAKE
        'verdict':   'REAL' if prob_real > 0.5 else 'FAKE',
    }

REAL = 1  →  sigmoid > 0.5  →  "실제 사진"

FAKE = 0  →  sigmoid < 0.5  →  "AI 생성 이미지"